# 实验六 P4 固定点真机分拣 修复版

修复“稳定帧 0/6 + ERROR”的误导显示。稳定确认、动作异常和 P4 等待清空分别处理。参数坐标、HSV 和 6 帧门槛沿用你的文件，不自动改动。

先停止旧预览并等待动作结束；建议重启当前 notebook 内核，然后按顺序运行本文件。点击启动预览会真实连接机械臂并返回拍摄位。


In [ ]:
# 现场调参区：只修改本单元即可调整摄像头、判色、P4 抓取和四桶上方位置。
import csv, math, sys, threading, time, traceback
from pathlib import Path
from datetime import datetime
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

BASE = Path('/home/jetson/Notebook/4.机械臂OpenCV视觉处理实验')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
CAMERA_ID = 0
ROI = (240, 210, 440, 390)
MIN_AREA = 700
CONFIRM_FRAMES = 6
POSITION_TOLERANCE = 18
EMPTY_REARM_FRAMES = 8
FRAME_DELAY = 0.04
MOVE_T = 1.5
GRIPPER_T = 0.5

# P4 的 Z=0.278 mm 是台面参考高度；XY/Z/Pitch 保留当前实测值。
# 参考 Week3 的正向放置 Roll=179.2°；抓取点与上方点使用同一角度，避免夹爪斜着下降。
P4 = (177.5, -9.0, 0.278)
P4_GRIP_ROLL_DEG = 179.2
P4_PICK_POSE = (193.2, -12.8, 28.0, 132.7, P4_GRIP_ROLL_DEG)
P4_APPROACH_POSE = (193.2, -12.8, 65.0, 132.7, P4_GRIP_ROLL_DEG)

# 坐标单位 mm，两个角度单位度；Roll 保留原始读数，运动时折算到 0～360 度。
bin_map = {
    'red': {'bin': '有害垃圾', 'pose': (127.4, -131.8, 176.4, 94.5, 148789.0)},
    'yellow': {'bin': '其他垃圾', 'pose': (219.0, 99.8, 175.6, 106.5, 137329.9)},
    'green': {'bin': '厨余垃圾', 'pose': (122.8, 114.8, 172.8, 112.7, -945200.4)},
    'blue': {'bin': '可回收垃圾', 'pose': (207.8, -141.3, 173.3, 104.5, 137329.9)},
}
HSV_RANGES = {
    'red': [((0, 80, 50), (10, 255, 255)), ((170, 80, 50), (179, 255, 255))],
    'yellow': [((18, 80, 60), (38, 255, 255))],
    'green': [((40, 60, 35), (90, 255, 255))],
    'blue': [((90, 75, 45), (135, 255, 255))],
}
BGR = {'red': (0, 0, 255), 'yellow': (0, 255, 255),
       'green': (0, 255, 0), 'blue': (255, 0, 0)}
CN = {'red': '红', 'yellow': '黄', 'green': '绿', 'blue': '蓝'}
KERNEL = np.ones((5, 5), np.uint8)
print('真机自动分拣；摄像头', CAMERA_ID, '；P4 判色区域', ROI)
# 动作后检查：只看回拍摄位之后的新帧，避免单帧误判。
POST_RETURN_SETTLE = 0.6
P4_CLEAR_TIMEOUT = 4.0
P4_CLEAR_FRAMES = 4
MAX_FRAME_AGE = 1.0
RESET_SETTLE = 0.5
MIN_SCENE_BRIGHTNESS = 10
ARM_WAIT_TIMEOUT = 5.0


## 1. 核对固定目标点与四个桶上方点

P4 抓取点与四桶位置均在上方“现场调参区”修改。P4 的 `P4_GRIP_ROLL_DEG` 控制夹爪正向角度，抓取与上方必须使用同一值。Roll 原始值是累计角，程序按 360° 周期折算后再发给机械臂。桶坐标是**松开夹爪的上方点**，不再下降到原 SDK 的桶底位姿。

In [ ]:
def normal_roll_deg(value):
    """将累计横滚角折算到 0～360 度。"""
    return float(value) % 360.0

print('P4 台面参考：', P4)
print('P4 抓取：', P4_PICK_POSE)
print('P4 上方：', P4_APPROACH_POSE)
for color, info in bin_map.items():
    x, y, z, pitch, roll = info['pose']
    print(f'{CN[color]}色 → {info["bin"]}：XYZ=({x:.1f}, {y:.1f}, {z:.1f}) mm，'
          f'Pitch={pitch:.1f}°，Roll 原始={roll:.1f}°，运动用={normal_roll_deg(roll):.1f}°')

## 2. HSV 实时判色

只在 P4 画面区域识别方块，避免后方其他彩色方块触发分拣。红色保留跨 H=0/179 的两段 HSV 阈值。每次只允许一个有效色块，连续稳定 6 帧后自动执行真机动作。

In [ ]:
def detect_color(frame, colors=None, roi=None):
    """返回有效目标列表、标注画面和四色掩码。"""
    if frame is None or frame.size == 0:
        raise ValueError('输入画面为空')
    chosen = list(bin_map) if colors is None else list(colors)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = (0, 0, width, height) if roi is None else roi
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(width, int(x2)), min(height, int(y2))
    if x1 >= x2 or y1 >= y2:
        raise ValueError('ROI 越界或为空')
    spatial = np.zeros((height, width), dtype=np.uint8)
    spatial[y1:y2, x1:x2] = 255
    canvas = frame.copy()
    detections, masks = [], {}
    for color in bin_map:
        mask = np.zeros((height, width), dtype=np.uint8)
        for lower, upper in HSV_RANGES[color]:
            mask |= cv2.inRange(hsv, np.array(lower, np.uint8), np.array(upper, np.uint8))
        mask = cv2.bitwise_and(mask, spatial)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
        masks[color] = mask
        if color not in chosen:
            continue
        contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < MIN_AREA:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            target = {'color': color, 'pixel': (x + w // 2, y + h // 2),
                      'box': (x, y, w, h), 'area': float(area)}
            detections.append(target)
            cv2.rectangle(canvas, (x, y), (x + w, y + h), BGR[color], 2)
            cv2.putText(canvas, f'{color} {area:.0f}', (max(5, x), max(25, y - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, .6, BGR[color], 2)
    if roi is not None:
        cv2.rectangle(canvas, (x1, y1), (x2, y2), (220, 220, 220), 1)
    return detections, canvas, masks

def mask_montage(masks):
    """把四色 Mask 排成一张便于截图的图。"""
    tiles = []
    for color in bin_map:
        tile = cv2.cvtColor(masks[color], cv2.COLOR_GRAY2BGR)
        cv2.putText(tile, color, (12, 28), cv2.FONT_HERSHEY_SIMPLEX,
                    .8, BGR[color], 2)
        tiles.append(tile)
    return cv2.vconcat([cv2.hconcat(tiles[:2]), cv2.hconcat(tiles[2:])])

## 3 动作与目标位姿预检

启动时检查 P4、回零、拍摄位和四色桶位。单个桶无逆解时，只阻止该颜色的抓取，并在面板列出 XYZ/Pitch/Roll。不会擅自降低桶位，以免碰撞桶沿。

真实动作失败会显示具体阶段和 Traceback。动作完成但 P4 没有连续清空时进入 WAIT_CLEAR，提示检查夹取，不再伪装成“稳定帧故障”，也不会重复抓取。


In [ ]:
from arm5dof_uservo import Arm5DoFUServo
from config import ARM_POSE_DICT

def normal_roll_deg(value):
    return float(value) % 360.0

def pose_from_values(values):
    x,y,z,pitch_deg,roll_deg=values
    if not np.all(np.isfinite([x,y,z,pitch_deg,roll_deg])):
        raise ValueError('位姿包含非有限数值')
    return ([float(x),float(y),float(z)],math.radians(float(pitch_deg)),math.radians(normal_roll_deg(roll_deg)))

def sdk_pose(name):
    x,y,z,pitch,roll=ARM_POSE_DICT[name]
    return ([float(x),float(y),float(z)],pitch,roll)

def set_motion_phase(name):
    with lock:
        runtime['phase']=name
        state=runtime['state']
        detail=runtime['detail']
    set_state(state,detail)

def checked_pose(arm,name,pose):
    xyz,pitch,roll=pose
    adjusted=arm.adjust_tool_posi(xyz)
    if not np.all(np.isfinite(adjusted)) or not math.isfinite(float(roll)):
        raise ValueError(name+' 坐标或角度非法')
    checked_pitch=arm.auto_gen_pitch(adjusted) if pitch is None else pitch
    if not math.isfinite(float(checked_pitch)):
        raise ValueError(name+' Pitch 非法')
    ok,_=arm.inverse_kinematic(adjusted,checked_pitch,roll)
    if not ok:
        raise ValueError('{} 逆解不可达：XYZ={}，Pitch={:.1f}°，Roll={:.1f}°。未执行该目标运动。'.format(
            name,xyz,math.degrees(checked_pitch),math.degrees(roll)))
    return pose

def checked_common_poses(arm):
    result={}
    for name,pose in [('抓取上方',pose_from_values(P4_APPROACH_POSE)),
                      ('P4 抓取点',pose_from_values(P4_PICK_POSE)),
                      ('回零点',sdk_pose('home')),('拍摄位姿',sdk_pose('capture_image'))]:
        set_motion_phase('预检/'+name)
        result[name]=checked_pose(arm,name,pose)
    return result

def precheck_bins(arm):
    blocked={}
    for color,info in bin_map.items():
        try:
            checked_pose(arm,color+' 桶上方',pose_from_values(info['pose']))
        except ValueError as exc:
            blocked[color]=str(exc)
    return blocked

def connect_arm():
    try:
        return Arm5DoFUServo(is_init_pose=False,config_folder=str(BASE/'config'))
    except SystemExit as exc:
        raise RuntimeError('机械臂初始化退出；检查舵机电源、串口和占用进程') from exc

def wait_arm_settled(arm,name):
    # The SDK soft trajectory returning is not by itself proof of arrival.
    deadline=time.monotonic()+ARM_WAIT_TIMEOUT
    while time.monotonic()<deadline:
        if arm.uservo.is_stop():
            return
        time.sleep(.03)
    raise RuntimeError(name+' 等待关节到位超时，请检查实际位置与反馈')

def move_to(arm,name,pose):
    set_motion_phase(name)
    xyz,pitch,roll=pose
    print('机械臂：{} → {}'.format(name,tuple(round(v,1) for v in xyz)))
    result=arm.set_tool_pose(xyz,tool_pitch=pitch,tool_roll=roll,
                           T=MOVE_T,is_soft_move=True,is_wait=True)
    if result is False:
        raise RuntimeError(name+' 运动失败，后续动作已停止')
    wait_arm_settled(arm,name)

def move_to_capture():
    set_motion_phase('启动/连接机械臂')
    arm=connect_arm()
    try:
        common=checked_common_poses(arm)
        blocked=precheck_bins(arm)
        with lock:
            runtime['blocked_colors']=blocked
        if blocked:
            ui_log('以下颜色桶位预检不通过，禁止这些颜色触发抓取：'+', '.join(blocked))
            for color,reason in blocked.items():
                ui_log(reason)
        update_preflight_label(blocked)
        move_to(arm,'拍摄位姿',common['拍摄位姿'])
    finally:
        arm.uart.close()

def run_pick_place(target):
    color=target['color']
    if color not in bin_map:
        raise ValueError('未知颜色：'+str(color))
    with lock:
        reason=runtime['blocked_colors'].get(color)
    if reason:
        raise ValueError(reason)
    set_motion_phase('连接机械臂')
    arm=connect_arm()
    try:
        poses=checked_common_poses(arm)
        set_motion_phase('预检/'+color+' 桶上方')
        poses['桶上方']=checked_pose(arm,color+' 桶上方',pose_from_values(bin_map[color]['pose']))
        set_motion_phase('张开夹爪')
        arm.gripper_open(T=GRIPPER_T,is_wait=True)
        wait_arm_settled(arm,'张开夹爪')
        move_to(arm,'抓取上方',poses['抓取上方'])
        move_to(arm,'P4 抓取点',poses['P4 抓取点'])
        set_motion_phase('闭合夹爪')
        arm.gripper_close(T=GRIPPER_T,is_wait=True)
        wait_arm_settled(arm,'闭合夹爪')
        move_to(arm,'抬起物块',poses['抓取上方'])
        move_to(arm,'回零转运',poses['回零点'])
        move_to(arm,'对应垃圾桶上方',poses['桶上方'])
        set_motion_phase('桶上方松爪')
        arm.gripper_open(T=GRIPPER_T,is_wait=True)
        wait_arm_settled(arm,'桶上方松爪')
        move_to(arm,'回零复位',poses['回零点'])
        move_to(arm,'恢复拍摄位姿',poses['拍摄位姿'])
        return '{}色 → {}：动作指令完成，桶内结果待人工确认'.format(CN[color],bin_map[color]['bin'])
    finally:
        # Never open or go Home again after an exception.
        arm.uart.close()

def verify_p4_cleared(frame,color):
    # Compatibility helper; the job uses wait_for_p4_clear() instead of one frame.
    if frame is None:
        return False
    return not detect_color(frame,None,ROI)[0]

def wait_for_p4_clear(returned_at):
    """Use successive NEW raw frames after return, not a cached annotated image."""
    deadline=time.monotonic()+P4_CLEAR_TIMEOUT
    cutoff=returned_at+POST_RETURN_SETTLE
    previous_sequence=-1
    empty_count=0
    last_colors=[]
    checked=0
    while time.monotonic()<deadline:
        with frame_condition:
            frame_condition.wait_for(lambda: runtime['stop'].is_set() or runtime['reader_error'] or
                                     runtime['capture_seq']>previous_sequence,timeout=.25)
            if runtime['stop'].is_set():
                raise RuntimeError('预览已停止，动作后无法核实 P4；检查现场后再启动')
            if runtime['reader_error']:
                raise RuntimeError('动作后摄像头异常：'+runtime['reader_error'])
            sequence=runtime['capture_seq']
            stamp=runtime['capture_time']
            frame=runtime['capture_frame']
            if frame is None or sequence<=previous_sequence:
                continue
            frame=frame.copy()
        previous_sequence=sequence
        if stamp<cutoff or time.monotonic()-stamp>MAX_FRAME_AGE:
            continue
        checked+=1
        detections,_,_=detect_color(frame,None,ROI)
        if not detections and frame_quality_ok(frame):
            empty_count+=1
        else:
            empty_count=0
        last_colors=sorted({item['color'] for item in detections})
        if empty_count>=P4_CLEAR_FRAMES:
            return dict(cleared=True,checked_frames=checked,last_colors=[])
    if checked==0:
        raise RuntimeError('回到拍摄位后未取得足够新鲜画面；请检查摄像头延迟/连接')
    return dict(cleared=False,checked_frames=checked,last_colors=last_colors)

## 4 实时预览

相机专用线程持续读取最新帧，视觉渲染与动作线程分别处理，减少缓冲旧画面。BUSY/VERIFY 期间禁止新任务，保留触发时的 6/6 计数。


In [ ]:
import html

# Release this notebook's previous workers before creating another panel.
if 'runtime' in globals():
    old_job=runtime.get('job_thread')
    if old_job is not None and old_job.is_alive():
        raise RuntimeError('上一次机械臂动作仍在执行；等结束后再重建面板')
    runtime['stop'].set()
    for name in ('thread','reader_thread'):
        old_thread=runtime.get(name)
        if old_thread is not None and old_thread.is_alive():
            old_thread.join(timeout=3)
            if old_thread.is_alive():
                raise RuntimeError('旧线程未退出：'+name+'；先停止旧预览，不可重复占用摄像头')

color_choice=widgets.Dropdown(options=[('四色自动识别','all'),('只识别红色','red'),
    ('只识别黄色','yellow'),('只识别绿色','green'),('只识别蓝色','blue')],value='all',description='颜色')
btn_start=widgets.Button(description='启动预览',button_style='success')
btn_snapshot=widgets.Button(description='保存实拍截图',button_style='info')
btn_stop=widgets.Button(description='停止预览',button_style='danger')
btn_reset=widgets.Button(description='处理后复位',button_style='warning')
state_label=widgets.HTML(value='状态：STOPPED')
error_label=widgets.HTML(value='')
preflight_label=widgets.HTML(value='桶位预检：启动时检查')
image_widget=widgets.Image(format='jpeg',width=640)
mask_widget=widgets.Image(format='jpeg',width=640)
log_box=widgets.Output(layout={'border':'1px solid #ddd','height':'200px','overflow':'auto'})
display(widgets.VBox([widgets.HBox([color_choice,btn_start,btn_snapshot,btn_stop,btn_reset]),
    state_label,error_label,preflight_label,widgets.HTML('<b>原画标注</b>'),image_widget,
    widgets.HTML('<b>四色 Mask</b>'),mask_widget,log_box]))

# New session directory; never append incompatible columns to an old CSV.
RUN_DIR=Path.cwd()/'week6_records'/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
ACTION_CSV=RUN_DIR/'week6_action_log.csv'
EVIDENCE_DIR=RUN_DIR/'evidence_week6'
lock=threading.RLock()
frame_condition=threading.Condition(lock)
runtime={'thread':None,'reader_thread':None,'job_thread':None,'stop':threading.Event(),
         'state':'STOPPED','detail':'','candidate':None,'stable':None,'count':0,
         'last_trigger_count':0,'empty':0,'frame_number':0,'pending':False,'latest':None,
         'last_job':None,'job_number':0,'mode':'真机','phase':'未开始','last_error':'',
         'blocked_colors':{},'capture_frame':None,'capture_time':0.0,'capture_seq':0,
         'reader_error':'','reset_after':0.0}

def ui_log(message):
    with log_box:
        print(datetime.now().strftime('%H:%M:%S'),message)

def set_state(state,detail=''):
    with lock:
        runtime['state']=state
        runtime['detail']=detail
        current=runtime['count']
        triggered=runtime['last_trigger_count']
        phase=runtime['phase']
    if state in ('BUSY','VERIFY','WAIT_CLEAR','ERROR','PARAM_ERROR'):
        count_text='上次触发确认 {}/{}（不是当前稳定帧故障）'.format(triggered,CONFIRM_FRAMES)
    else:
        count_text='视觉稳定帧 {}/{}'.format(current,CONFIRM_FRAMES)
    state_label.value='<b>状态：{}</b>　{}　阶段：{}　{}'.format(html.escape(state),count_text,html.escape(phase),html.escape(detail))

def show_error(message):
    with lock:
        runtime['last_error']=message
    error_label.value='<div style="color:#b00020;white-space:pre-wrap"><b>具体原因：</b>'+html.escape(message)+'</div>'

def update_preflight_label(blocked):
    if not blocked:
        preflight_label.value='<span style="color:green">四色桶位 IK 预检通过（不代表路径无碰撞）。</span>'
    else:
        preflight_label.value='<div style="color:#9a5800;white-space:pre-wrap"><b>禁止抓取的颜色：</b>\n'+html.escape('\n'.join(blocked.values()))+'</div>'

def append_csv(path,fields,row):
    path.parent.mkdir(parents=True,exist_ok=True)
    exists=path.exists() and path.stat().st_size>0
    with path.open('a',newline='',encoding='utf-8-sig') as stream:
        writer=csv.DictWriter(stream,fieldnames=fields)
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def matching_target(a,b):
    return bool(a and b and a['color']==b['color'] and
                math.hypot(a['pixel'][0]-b['pixel'][0],a['pixel'][1]-b['pixel'][1])<=POSITION_TOLERANCE)

def frame_quality_ok(frame):
    h,w=frame.shape[:2]
    x1,y1,x2,y2=(0,0,w,h) if ROI is None else ROI
    crop=frame[max(0,y1):min(h,y2),max(0,x1):min(w,x2)]
    return crop.size>0 and float(cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY).mean())>=MIN_SCENE_BRIGHTNESS

def camera_reader(capture):
    """Only this thread reads/releases the camera, continuously draining buffers."""
    try:
        while not runtime['stop'].is_set():
            ok,frame=capture.read()
            if not ok or frame is None:
                raise RuntimeError('摄像头取帧失败')
            with frame_condition:
                runtime['capture_frame']=frame.copy()
                runtime['capture_time']=time.monotonic()
                runtime['capture_seq']+=1
                frame_condition.notify_all()
    except Exception as exc:
        with frame_condition:
            runtime['reader_error']=str(exc)
            frame_condition.notify_all()
    finally:
        capture.release()

def run_job(target):
    error=''
    failed_phase=''
    clear_result=None
    with lock:
        trigger_count=runtime['last_trigger_count']
    try:
        message=run_pick_place(target)
        returned=time.monotonic()
        set_motion_phase('动作后连续新帧检查 P4')
        set_state('VERIFY','等待拍摄位稳定和新帧清空确认')
        clear_result=wait_for_p4_clear(returned)
        if clear_result['cleared']:
            ui_log(message+'；连续新帧确认 P4 已清空')
        else:
            ui_log('动作流程已返回，但 P4 未连续清空。可能未夹住/物块掉回/新物块放入过早；请人工检查，不会重复抓取。')
    except Exception as exc:
        error=type(exc).__name__+': '+str(exc)
        with lock:
            failed_phase=runtime['phase']
        detail='阶段 [{}]：{}'.format(failed_phase,error)
        show_error(detail)
        ui_log('动作异常：'+detail)
        trace=traceback.format_exc()
        with log_box:
            print(trace)
        RUN_DIR.mkdir(parents=True,exist_ok=True)
        (RUN_DIR/('error_'+datetime.now().strftime('%H%M%S_%f')+'.txt')).write_text(trace,encoding='utf-8')
    with lock:
        runtime['job_number']+=1
        cleared=bool(clear_result and clear_result['cleared'])
        status='动作异常' if error else ('P4 已清空，桶内待人工确认' if cleared else '动作返回，P4 未清空，待人工处理')
        job={'编号':runtime['job_number'],'时间':datetime.now().isoformat(timespec='seconds'),
             '识别颜色':target['color'],'目标桶':bin_map[target['color']]['bin'],'像素坐标':str(target['pixel']),
             '模式':'真机','触发确认帧':trigger_count,'执行状态':status,'失败阶段':failed_phase,
             'P4清空':cleared,'清空检查帧':0 if clear_result is None else clear_result['checked_frames'],'异常':error}
        runtime['last_job']=job
        runtime.update(candidate=None,stable=None,count=0,empty=0,pending=False,reset_after=time.monotonic()+RESET_SETTLE)
        if error:
            set_state('ERROR','动作已禁止；具体错误见上方红字及下方日志')
        elif runtime['stop'].is_set():
            set_state('STOPPED','动作已结束，预览已停止')
        elif runtime['state']=='ERROR':
            pass
        elif cleared:
            set_state('RESET','等待 P4 连续空区，再接收下一块')
        else:
            set_state('WAIT_CLEAR','P4 仍有目标；人工检查夹取并清空，不会重复抓取')
    try:
        append_csv(ACTION_CSV,list(job),job)
    except Exception as exc:
        show_error('动作结束但日志写入失败：'+str(exc))
        set_state('ERROR','检查结果目录写权限')

def preview_loop():
    capture=None
    reader=None
    try:
        capture=cv2.VideoCapture(CAMERA_ID)
        if not capture.isOpened():
            raise RuntimeError('摄像头 {} 打开失败'.format(CAMERA_ID))
        reader=threading.Thread(target=camera_reader,args=(capture,),daemon=True)
        with lock:
            runtime['reader_thread']=reader
        reader.start()
        with frame_condition:
            frame_condition.wait_for(lambda: runtime['capture_frame'] is not None or runtime['reader_error'],timeout=2)
            if runtime['reader_error']:
                raise RuntimeError(runtime['reader_error'])
            if runtime['capture_frame'] is None:
                raise RuntimeError('摄像头打开但两秒内没有输出画面，未移动机械臂')
        move_to_capture()
        with lock:
            runtime.update(empty=0,count=0,candidate=None,stable=None,pending=False)
            runtime['phase']='等待 P4'
        set_state('READY','放一个物块在 P4；稳定后自动真机分拣')
        previous_choice=color_choice.value
        last_sequence=-1
        while not runtime['stop'].is_set():
            with frame_condition:
                frame_condition.wait_for(lambda: runtime['stop'].is_set() or runtime['reader_error'] or
                                         runtime['capture_seq']>last_sequence,timeout=1)
                if runtime['stop'].is_set():
                    break
                if runtime['reader_error']:
                    raise RuntimeError(runtime['reader_error'])
                sequence=runtime['capture_seq']
                if sequence<=last_sequence:
                    raise RuntimeError('摄像头超过 1 秒没有新帧')
                raw=runtime['capture_frame'].copy()
                stamp=runtime['capture_time']
            last_sequence=sequence
            if time.monotonic()-stamp>MAX_FRAME_AGE:
                raise RuntimeError('相机画面已过期，禁止用旧画面触发抓取')
            choice=color_choice.value
            if choice!=previous_choice:
                with lock:
                    runtime.update(candidate=None,stable=None,count=0,pending=False)
                previous_choice=choice
            colors=None if choice=='all' else [choice]
            detections,canvas,masks=detect_color(raw,colors,ROI)
            all_detections=detections if choice=='all' else detect_color(raw,None,ROI)[0]
            single=detections[0] if len(detections)==1 and len(all_detections)==1 and frame_quality_ok(raw) else None
            target=None
            with lock:
                state=runtime['state']
                if state in ('RESET','WAIT_CLEAR','PARAM_ERROR'):
                    empty=(not all_detections and frame_quality_ok(raw) and time.monotonic()>=runtime['reset_after'])
                    runtime['empty']=runtime['empty']+1 if empty else 0
                    if runtime['empty']>=EMPTY_REARM_FRAMES:
                        runtime.update(candidate=None,stable=None,count=0,pending=False,empty=0)
                        runtime['phase']='等待 P4'
                        set_state('READY','P4 已清空，可放入下一块')
                elif state in ('READY','DETECT','CONFIRM'):
                    if single and matching_target(runtime['candidate'],single):
                        runtime['count']=min(CONFIRM_FRAMES,runtime['count']+1)
                    else:
                        runtime['candidate']=None if single is None else dict(single)
                        runtime['count']=1 if single else 0
                    runtime['stable']=dict(single) if single and runtime['count']>=CONFIRM_FRAMES else None
                    if runtime['stable']:
                        runtime['last_trigger_count']=runtime['count']
                        blocked=runtime['blocked_colors'].get(single['color'])
                        if blocked:
                            show_error(blocked)
                            runtime['pending']=False
                            set_state('PARAM_ERROR','颜色已稳定，但桶位参数不可达；未启动抓取。移走物块后可识别其他颜色。')
                        else:
                            target=dict(runtime['stable'])
                            runtime['pending']=False
                            runtime['phase']='准备抓取'
                            error_label.value=''
                            runtime['last_error']=''
                            set_state('BUSY','持续预览；禁止启动第二个动作')
                    else:
                        set_state('DETECT' if all_detections else 'READY','仅允许一个目标' if len(all_detections)>1 else '')
                runtime['frame_number']+=1
                text='{} frame {} confirmed={}/{}'.format(runtime['state'],sequence,
                      runtime['last_trigger_count'] if runtime['state'] in ('BUSY','VERIFY','WAIT_CLEAR','ERROR','PARAM_ERROR') else runtime['count'],CONFIRM_FRAMES)
            cv2.putText(canvas,text,(12,28),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
            montage=mask_montage(masks)
            with lock:
                runtime['latest']=(raw.copy(),canvas.copy(),montage.copy())
            ok,jpeg=cv2.imencode('.jpg',canvas,[cv2.IMWRITE_JPEG_QUALITY,80])
            if ok:
                image_widget.value=jpeg.tobytes()
            ok,jpeg=cv2.imencode('.jpg',montage,[cv2.IMWRITE_JPEG_QUALITY,70])
            if ok:
                mask_widget.value=jpeg.tobytes()
            if target:
                job=threading.Thread(target=run_job,args=(target,),daemon=True)
                with lock:
                    runtime['job_thread']=job
                job.start()
            runtime['stop'].wait(FRAME_DELAY)
    except Exception as exc:
        ui_log('预览/启动异常：'+str(exc))
        show_error('阶段 [{}]：{}: {}'.format(runtime['phase'],type(exc).__name__,exc))
        set_state('ERROR',str(exc))
    finally:
        runtime['stop'].set()
        with frame_condition:
            frame_condition.notify_all()
        if reader is not None:
            reader.join(timeout=3)
            if reader.is_alive():
                ui_log('相机读取仍未退出，稍后再启动，勿重复占用设备')
        elif capture is not None:
            capture.release()
        with lock:
            runtime['pending']=False
            job=runtime['job_thread']
            if runtime['state']!='ERROR' and not (job is not None and job.is_alive()):
                set_state('STOPPED','摄像头已释放')

def on_start(_):
    with lock:
        for name in ('thread','reader_thread','job_thread'):
            job=runtime.get(name)
            if job is not None and job.is_alive():
                ui_log('已有线程运行：'+name+'；先停止并等待，不会重复连接设备')
                return
        runtime['stop']=threading.Event()
        runtime.update(candidate=None,stable=None,count=0,last_trigger_count=0,pending=False,latest=None,
                       capture_frame=None,capture_time=0.0,capture_seq=0,reader_error='',
                       frame_number=0,mode='真机',phase='启动',last_error='',blocked_colors={})
        error_label.value=''
        set_state('STARTING','检查目标位姿并返回拍摄位')
        runtime['thread']=threading.Thread(target=preview_loop,daemon=True)
        runtime['thread'].start()
    ui_log('启动真机预览；稳定帧是视觉确认，动作异常将单独显示阶段与原因')

def on_reset(_):
    with lock:
        job=runtime['job_thread']
        if job is not None and job.is_alive():
            ui_log('机械臂动作/清空检查未结束，不能复位')
            return
        if runtime['stop'].is_set() or runtime['capture_frame'] is None:
            ui_log('请先启动预览；若改了位姿，停止并重新启动以重做预检')
            return
        runtime.update(candidate=None,stable=None,count=0,empty=0,pending=False,last_error='',reset_after=time.monotonic()+RESET_SETTLE)
        error_label.value=''
        runtime['phase']='人工处理后复位'
        set_state('RESET','先清空 P4；不会对仍在原处的物块直接重试')

def on_snapshot(_):
    with lock:
        if runtime['latest'] is None:
            ui_log('没有真实画面可保存')
            return
        pictures=tuple(image.copy() for image in runtime['latest'])
    EVIDENCE_DIR.mkdir(parents=True,exist_ok=True)
    stamp=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    for label,picture in zip(('原图','识别','四色Mask'),pictures):
        ok,encoded=cv2.imencode('.png',picture)
        if not ok:
            raise IOError('图片编码失败')
        encoded.tofile(str(EVIDENCE_DIR/(stamp+'_'+label+'.png')))
    ui_log('已保存：'+str(EVIDENCE_DIR/stamp))

def on_stop(_):
    runtime['stop'].set()
    with frame_condition:
        runtime['pending']=False
        frame_condition.notify_all()
    ui_log('已请求停止预览和新任务；正在执行的 SDK 动作仍需等待或使用设备急停')

btn_start.on_click(on_start)
btn_snapshot.on_click(on_snapshot)
btn_stop.on_click(on_stop)
btn_reset.on_click(on_reset)
print('本次记录目录：',RUN_DIR)

## 5 如何处理提示

- READY/DETECT：正在等待或累计稳定帧；只有一个目标且位置稳定才触发。
- BUSY/VERIFY：已确认颜色并执行动作，随后连续检查回拍摄位后的新帧。
- WAIT_CLEAR：动作已返回，但 P4 仍有物体/无法连续判为空区。检查夹爪、抓取高度和实际物块，不会自动重抓。清空 P4 后连续 8 帧即可恢复。
- PARAM_ERROR：该颜色已识别稳定，但桶位 IK 失败，未启动抓取。按上方列出的参数核对位姿，修改后停止并重新启动预览以重做预检。
- ERROR：查看界面红字的具体原因、失败阶段和底部 Traceback；完整错误也写入本次 week6_records 目录。处理后点击“处理后复位”，先清空 P4 再接收新任务。

保留原文件中的桶位和 P4 参数。按本地现有 SDK 离线检查，当前黄/绿/蓝桶被抬高后的 Z 分别为 175.6/172.8/173.3 mm，没有逆解；红桶 176.4 mm 有解。现场 SDK/标定不同时以启动预检为准。有 IK 解不代表路径无碰撞，不能直接把桶高降下来绕过问题。

输出新建于当前目录 week6_records/每次运行时间/，保留旧日志。动作 CSV 区分动作异常与 P4 未清空，不把程序返回当作投放成功。停止预览不能硬中断正在执行的旧 SDK 运动；紧急情况使用设备急停。
